# Ocena na test skupu

Validaciju modeli nisu koristili jednako, pa njihove dosadašnje brojke nisu uporedive. U ovoj svesci se modeli samo učitavaju i ocenjuju, bez treniranja.

Skup podataka se može preuzeti sa [ove](https://www.kaggle.com/datasets/nathanlauga/nba-games) adrese.

## Priprema

In [1]:
import sys
sys.path.append("..")

import joblib
import numpy as np
import pandas as pd

import torch
from torch.utils.data import DataLoader

from src.config import BATCH_SIZE, MODELS_PATH, PROCESSED_DATA_PATH, RANDOM_STATE, RESULTS_FILE
from src import data, evaluate, models, plotting, sequences, train

np.random.seed(RANDOM_STATE)

## Učitavanje podataka

Isti podaci kao u sveskama 03 i 04, pa su mečevi isti za sve modele.

Test je jedna sezona, 2021/22. Pandemijske sezone su izbačene iz svih skupova, pa između validacije i testa stoji razmak od 859 dana; u sekvence i dalje ulaze kao istorija, ali se ne predviđaju.

In [2]:
X = pd.read_csv(PROCESSED_DATA_PATH / "atributi.csv")
splits = np.load(PROCESSED_DATA_PATH / "sekvence.npz")
y = splits["labels"]

test = splits["test"]
X_test, y_test = X.iloc[test], y[test]

# GAME_ID ne stoji u sekvence.npz, pa se sirovi skup ucitava samo zbog njega
df_games = data.clean_games(data.load_games())
game_ids = df_games.loc[splits["indices"][test], "GAME_ID"].to_numpy()

dataset_test = sequences.MatchSequenceDataset(
    splits["home_sequences"][test], splits["away_sequences"][test],
    splits["context"][test], y_test,
)
loader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

len(dataset_test), X_test.shape

(1323, (1323, 52))

Udeo pobeda domaćina na validaciji i na testu. Ako se razlikuje, i metrike se teže porede.

In [3]:
float(y[splits["validation"]].mean()), float(y_test.mean())

(0.5887957215309143, 0.547996997833252)

## Učitavanje sačuvanih modela

Klasični modeli su cele `joblib` datoteke, a mreže su `state_dict` - samo težine, pa se pre učitavanja mora napraviti `MatchPredictor` istog oblika. `hidden_size` se čita iz samih težina, pa sveska ne mora da zna šta je pretraga izabrala u svesci 04.

Svaka mreža ima pet sačuvanih modela, po jedan za svako seme.

In [4]:
KLASICNI = {
    "M1 log. reg.": "log_reg.joblib",
    "M2 XGBoost": "m2_xgboost.joblib",
}

MREZE = {
    "M3 RNN": ("rnn", "rnn"),
    "M4 LSTM": ("lstm", "lstm"),
    "M5 GRU": ("gru", "gru"),
}


def snimci(osnova):
    """Svi sacuvani modeli jedne mreze, po jedan za svako seme."""
    return sorted(MODELS_PATH.glob(f"{osnova}_seed*.pt"))


# bez ove provere bi mreza bez sacuvanih modela bila tiho preskocena, pa bi
# tabela imala manje redova a sveska ne bi pukla
assert all(snimci(osnova) for osnova, _ in MREZE.values()), "nedostaju sacuvani modeli"


def ucitaj_mrezu(putanja, cell):
    """Rekonstruise mrezu iz tezina; hidden_size cita iz njih samih."""
    state = torch.load(putanja, weights_only=True)

    model = models.MatchPredictor(
        n_features=splits["home_sequences"].shape[-1],
        n_context=splits["context"].shape[1],
        cell=cell,
        hidden_size=state["encoder.weight_hh_l0"].shape[1],
    )
    model.load_state_dict(state)
    return model


{ime: ucitaj_mrezu(snimci(osnova)[0], cell).encoder.hidden_size
 for ime, (osnova, cell) in MREZE.items()}

{'M3 RNN': 64, 'M4 LSTM': 32, 'M5 GRU': 32}

## Ocena na test skupu

M0 nema sačuvan model; predviđa udeo pobeda domaćina na treningu, isto kao u svesci 03. Udeo se uzima sa treninga, ne sa testa, inače bi znao nešto o skupu koji tek treba da vidi.

Kod mreža se svih pet modela ocenjuje zasebno: u `rezultati.csv` ide prosek, a pojedinačne vrednosti u `rasipanje-semena.csv`.

In [5]:
base_rate = y[splits["train"]].mean()

# float64 svuda: mreze i XGBoost vracaju float32, pa bi metrike odstupale od
# istih metrika racunatih iz sacuvanog fajla
predvidjanja = {
    "GAME_ID": game_ids,
    "y_true": np.asarray(y_test, dtype="float64"),
    "M0 domacin": np.full(len(y_test), base_rate, dtype="float64"),
}

for ime, file_name in KLASICNI.items():
    y_proba = joblib.load(MODELS_PATH / file_name).predict_proba(X_test)[:, 1]
    predvidjanja[ime] = np.asarray(y_proba, dtype="float64")

proseci = {}

for ime, (osnova, cell) in MREZE.items():
    po_semenu, metrike_po_semenu = {}, []

    for putanja in snimci(osnova):
        y_proba, _ = train.predict_proba(ucitaj_mrezu(putanja, cell), loader_test)
        y_proba = np.asarray(y_proba, dtype="float64")
        seed = int(putanja.stem.split("_seed")[-1])

        metrike = evaluate.compute_metrics(predvidjanja["y_true"], y_proba)
        evaluate.append_seed_results(ime, "test", seed, metrike)
        po_semenu[seed] = y_proba
        metrike_po_semenu.append(metrike)

    predvidjanja[ime] = po_semenu[RANDOM_STATE]
    proseci[ime] = pd.DataFrame(metrike_po_semenu).mean().to_dict()

for ime in plotting.MODEL_ORDER:
    if ime not in proseci:
        proseci[ime] = evaluate.compute_metrics(predvidjanja["y_true"], predvidjanja[ime])
        evaluate.append_seed_results(ime, "test", RANDOM_STATE, proseci[ime])

    evaluate.append_results(ime, "test", proseci[ime])

pd.read_csv(RESULTS_FILE).query("skup == 'test'").round(4)

,model,skup,tacnost,roc_auc,log_loss,brier
12,M0 domacin,test,0.5480,0.5000,0.6940,0.2504
13,M1 log. reg.,test,0.6417,0.6805,0.6518,0.2288
14,M2 XGBoost,test,0.6395,0.6789,0.6487,0.2275
15,M3 RNN,test,0.6499,0.6791,0.6444,0.2257
16,M4 LSTM,test,0.6444,0.6807,0.6444,0.2259
17,M5 GRU,test,0.6402,0.6793,0.6447,0.2260


Prosek ne kaže koliko se pojedinačni prolazi razlikuju. Raspon po semenima to pokazuje.

In [6]:
po_semenu = pd.read_csv(evaluate.SEED_RESULTS_FILE).query("skup == 'test'")

raspon = po_semenu.groupby("model")["tacnost"].agg(["min", "mean", "max"])
raspon["raspon"] = raspon["max"] - raspon["min"]

raspon.reindex(plotting.MODEL_ORDER).round(4)

,min,mean,max,raspon
model,,,,
M0 domacin,0.5480,0.5480,0.5480,0.0000
M1 log. reg.,0.6417,0.6417,0.6417,0.0000
M2 XGBoost,0.6395,0.6395,0.6395,0.0000
M3 RNN,0.6425,0.6499,0.6538,0.0113
M4 LSTM,0.6379,0.6444,0.6508,0.0128
M5 GRU,0.6364,0.6402,0.6463,0.0098


Kod klasičnih modela raspon je nula, jer nemaju random inicijalizaciju. Kod mreža je između 0.98 i 1.28 procentnih poena.

## Test naspram validacije

Kolona `izlozenost` kaže koliko je koji model koristio validaciju pri izboru.

In [7]:
rezultati = pd.read_csv(RESULTS_FILE)

IZLOZENOST = {
    "M0 domacin": "nikakva",
    "M1 log. reg.": "nikakva",
    "M2 XGBoost": "pretraga",
    "M3 RNN": "pretraga i rano zaustavljanje",
    "M4 LSTM": "pretraga i rano zaustavljanje",
    "M5 GRU": "pretraga i rano zaustavljanje",
}


def uporedi(metrika):
    """Validacija naspram testa za jednu metriku, po dogovorenom redosledu."""
    tabela = rezultati.pivot(index="model", columns="skup", values=metrika)
    tabela = tabela.reindex(plotting.MODEL_ORDER)[["validacija", "test"]]
    tabela["razlika"] = tabela["test"] - tabela["validacija"]
    return tabela


tacnost = uporedi("tacnost").round(4)
tacnost.insert(0, "izlozenost", pd.Series(IZLOZENOST))
tacnost

skup,izlozenost,validacija,test,razlika
model,,,,
M0 domacin,nikakva,0.5888,0.5480,-0.0408
M1 log. reg.,nikakva,0.6562,0.6417,-0.0145
M2 XGBoost,pretraga,0.6570,0.6395,-0.0176
M3 RNN,pretraga i rano zaustavljanje,0.6579,0.6499,-0.0080
M4 LSTM,pretraga i rano zaustavljanje,0.6566,0.6444,-0.0121
M5 GRU,pretraga i rano zaustavljanje,0.6579,0.6402,-0.0176


In [8]:
uporedi("roc_auc").round(4)

skup,validacija,test,razlika
model,,,
M0 domacin,0.5000,0.5000,0.0000
M1 log. reg.,0.6953,0.6805,-0.0149
M2 XGBoost,0.6990,0.6789,-0.0201
M3 RNN,0.6968,0.6791,-0.0177
M4 LSTM,0.6969,0.6807,-0.0162
M5 GRU,0.6975,0.6793,-0.0182


Svi modeli padaju, između 0.8 i 1.8 procentnih poena, ali pad ne prati izloženost validaciji. M1 je uopšte nije koristio a pao je 1.45, više od M3 (0.80) i M4 (1.21), koji su je koristili najviše. 

Pad je stvar skupa, ne modela: domaćin je pobedio u 58.88% mečeva validacije, a u 54.80% na testu. Zato M0 pada najviše, 4.08 poena - on ne uči ništa osim da uvek kaže „domaćin".

## Čuvanje predviđanja

Verovatnoće se čuvaju da bi kasnija analiza - ROC krive, kalibracija, matrice konfuzije - radila iz jednog izvora. Kad bi se modeli svaki put učitavali iznova, brojke bi mogle tiho da se raziđu, jer test metrike zavise od `atributi.csv`, koji nije u gitu.

Za mreže se čuva prolaz sa semenom `RANDOM_STATE`, po jedan po modelu; rasipanje stoji u `rasipanje-semena.csv`.

In [9]:
PREDVIDJANJA_FILE = RESULTS_FILE.parent / "predvidjanja-test.csv"

pd.DataFrame(predvidjanja).to_csv(PREDVIDJANJA_FILE, index=False)

pd.read_csv(PREDVIDJANJA_FILE).head(3).round(4)

,GAME_ID,y_true,M0 domacin,M1 log. reg.,M2 XGBoost,M3 RNN,M4 LSTM,M5 GRU
0,22100002,0.0,0.5995,0.6890,0.6143,0.6050,0.6693,0.4783
1,22100001,1.0,0.5995,0.7870,0.6751,0.6452,0.7265,0.5073
2,22100005,1.0,0.5995,0.6643,0.6541,0.5473,0.6446,0.4940


## Zaključak

Na testu su svi modeli između 0.6395 i 0.6499. Najbolji je M3 RNN, najlošiji M2 XGBoost, a između njih je 1.04 procentna poena.

Ta razlika je istog reda kao i razlika između dva pokretanja iste mreže: kroz pet semena raspon ide od 0.98 do 1.28 poena. Kad se dve veličine toliko preklapaju, poredak ne razdvaja modele.

Isto pokazuje i ROC-AUC: sve vrednosti su između 0.6789 i 0.6807, ali je tu prvi M4, a ne M3.

Uz to, mreže nisu podešavane isto - M3 ima 64 jedinice, M4 i M5 po 32 - pa se ni ne razlikuju samo po tipu ćelije.

Glavni nalaz projekta se potvrdio: rekurentne mreže se ne izdvajaju od klasičnih modela, a M5 GRU je čak i ispod M1.

M0 predviđa samo da domaćin pobeđuje, pa mu tačnost prati udeo pobeda domaćina - kad je taj udeo pao sa 58.88% na 54.80%, M0 je pao za isto toliko, 4.08 poena. Ostali modeli gledaju i Elo i stanje na tabeli, pa ih ta promena pogađa manje.